# Lab 8 — Connect the five stages
### EP0418 · Topic 8 · 2 practical hours

**You need:** your group's track number (chosen at Lab 7; not sure? use 2, the
classroom sensors). The notebook fetches every file itself. Have the printed Lab 8
sheet beside you.

---
You have built every piece separately. Today you connect them — and the joins are
where systems actually break.

## The method: skeleton first

**Do not** connect five finished components and hope. Build the whole chain with
everything faked, get it running end to end, then replace one stage at a time.

If you connect five untested things and it fails, you have five suspects.

## How the cells work

- **Grey code cells:** run each one when you reach it, top to bottom. Click in it
  and press Shift+Enter.
- **Text boxes** have a bold label ending in a colon, like **My answers:**.
  Double-click, type, press Shift+Enter. Or write on the printed sheet.
- **Gemini's code** arrives as a **green cell**. Click **Accept and run** (bottom of
  the chat panel), then run the task's check cell.
- **Check cells** start with `# TODO(you):`, which says what to run first. Never
  change them: if Gemini shows red and green lines inside one, click **Cancel**
  (red and green lines inside Gemini's own green cell are fine). Leave lines like
  `s = keep("s")` as they are. If a check cell says *Not there yet* or *No ... yet*,
  type in the chat *call the result* and the name the task asks for, accept, and
  run the check cell again.
- **Gemini's lab rules** (*EP0418 lab rules*) are built in. If a box *Review custom
  instructions* appears, click **Use custom instruction**. The sliders icon beside
  **+** in the chat box should be blue; if it is grey, click it and choose *EP0418
  lab rules*. Keep *Learn mode* off.
- **Gemini used up for today** (*Daily limit reached*)? Paste the same prompt into
  ChatGPT, Claude or Copilot in a new tab. It cannot see your notebook: copy its code
  into a new cell (**+ Code** at the top of the page) and run it, and paste any
  failed check or red error back into its chat. Do the quiz last to save your limit.
- **Fill-in lines** are code lines waiting for your answer. Change only what comes
  after the `=`, then run the cell:
  - text goes between the quotes: `NAME = ""` becomes `NAME = "Tan Wei Ling"`;
  - a number replaces `None`: `H = None` becomes `H = 6`. Write numbers without
    commas: `128000`, not `128,000`.

---
## How every lab in this module works

Your AI assistant writes the code. Your job is what it cannot do: decide what the
right answer looks like, read what it gave you, and prove it did what you asked.

- **Prompts** in grey quote boxes start with **Rules:** (for example, do not drop
  rows; show the row count before and after), then **Task:**. Keep the Rules when
  you change a prompt. A prompt with no example is **zero-shot**; one example of the
  output you want makes it **one-shot**, several make it **few-shot**.
- **Checks** (the `check.*` lines under a task) are the **verification contract**.
  Do not edit them. In industry they are called **evals**.
- **A failing check** says what it found and what to try. Failing checks are normal
  while you work.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. If a box *Review custom instructions* appears while you save the copy, click **Make a copy**: it keeps the lab's short rules for the AI assistant. Colab opens your copy in a new tab with *Copy of* at the start of its name at the top left: work in that tab and close the old one. Reopen your copy from Google Drive next time.

> **Then run the grey code cell just below** (click in it and press Shift+Enter, or click the play button at its left). Colab may warn that the notebook was not authored by Google: click **Run anyway**. You should see nothing in red; lines like *fetched ...*, a row count or a short table are normal. That means the course files and tools are loaded. Every lab starts this way. From here on, **run every grey code cell when you reach it**, top to bottom. **To write in a text box** (a cell with a bold label ending in a colon), double-click it, type, then press Shift+Enter.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def keep(name):
    """The value Gemini's cell already made under this name, or ... if there is none yet."""
    global _NAMED
    v = globals().get(name, ...)
    if v is ... or v is None:
        print(f"Not there yet: nothing called {name}. Not run Gemini's green cell yet? Click Accept and "
              f"run on it, then run this cell again. Already ran it? Type in the same chat: call the "
              f"result {name}. Then Accept and run, and run this cell again.")
        _NAMED = True
        return ...
    return v


def filled(*values):
    """True once every TODO above has a value. Otherwise says what to do."""
    global _NAMED
    if any(v is ... or v is None for v in values):
        if not globals().get("_NAMED"):      # keep() has already said exactly what is missing
            print("Not there yet. Run Gemini's code first (click Accept and run on its green cell, "
                  "or paste its code into a new cell and run it), then run this cell again.")
        _NAMED = False
        return False
    _NAMED = False
    return True


def show_predict(data, unit):
    """Prediction League: run the newest predict that Gemini wrote and print its code to copy."""
    ran = globals().get("In", [])                                   # every cell you have run, oldest first
    mine = [c for c in ran if "def predict(" in c and "The baseline predicts" not in c and "show_predict" not in c]
    if not mine:
        print("Gemini's predict has not run yet. Find the cell Gemini added in Step 2 (it starts with")
        print("def predict, or with import lines just above it), click in it, press Shift+Enter,")
        print("then run this cell again."); return
    code = mine[-1].strip()
    for short, line in (("np.", "import numpy as np"), ("pd.", "import pandas as pd")):
        if short in code and line not in code:                     # the league page starts empty
            code = line + "\n" + code
    ns = {}
    exec(code, ns)
    print("Your predict says", round(ns["predict"](data.copy()), 1), unit)
    print()
    print("----- copy from the line below -----")
    print(code)
    print("----- to the line above -----")

## Task 1 — The skeleton, entirely faked · `VERIFY`

The five stages are five small functions: ingest (load), clean, analyse (detect),
decide (rule), respond (send). The cell below defines all five with fake insides
and runs them as a chain. It does nothing useful. That is the point. Tags show where you are in the loop: `PROMPT` → `GENERATE` → `VERIFY` → `REFINE`.

1. `VERIFY` **Run the cell below.**
2. `VERIFY` **Read what it prints**: one line per stage, plus a line saying which stages
   are still the fake skeleton (all of them, for now).

In [ ]:
def ingest():
    """Stage 1 — later: read your real data."""
    return pd.DataFrame({"timestamp": pd.date_range("2026-06-01", periods=10, freq="h"),
                         "value": range(10)})

def clean(df):
    """Stage 2 — later: your real cleaning."""
    return df

def analyse(df):
    """Stage 3 — later: your detector or forecast."""
    return df.assign(flag=False)

def decide(df):
    """Stage 4 — later: your rule, debounced and rate-limited."""
    return df[df.flag]

def respond(events):
    """Stage 5 — later: your real notification."""
    print(f"would send {len(events)} alert(s)")
    return [f"alert {i}" for i in range(len(events))]

for _f in (ingest, clean, analyse, decide, respond):
    _f.fake = True          # marks the skeleton stages; Gemini's versions replace them

def pipeline():
    still_fake = [f.__name__ for f in (ingest, clean, analyse, decide, respond) if getattr(f, "fake", False)]
    if still_fake:
        print("  (still the fake skeleton: " + ", ".join(still_fake) + ")")
    df = ingest()
    print(f"  ingest  -> {len(df):,} rows")
    df = clean(df)
    print(f"  clean   -> {len(df):,} rows")
    df = analyse(df)
    print(f"  analyse -> {len(df):,} rows")
    events = decide(df)
    print(f"  decide  -> {len(events):,} events")
    return respond(events)

sent = pipeline()

**What you should see:** `ingest -> 10 rows`, `clean -> 10 rows`, `analyse -> 10
rows`, `decide -> 0 events`, and `would send 0 alert(s)`.

### Note the row count printed at every stage

That is not decoration. **"Zero alerts" and "a stage is silently producing
nothing" look identical from outside.** Printing the shape at each stage is how
you tell them apart — in this lab and in your CA3 report.

## Task 2 — Replace one stage at a time · `PROMPT` `GENERATE` `VERIFY` `REFINE`

You swap the fake stages for real ones, one at a time, and **run the whole chain
after every swap**. When it breaks you will know exactly which change broke it.

**Stages 1 and 2 (ingest and clean) come ready-made** for every track, in
`track_kit.py`, each handling that file's known faults. You choose your track and
read them; you do not write them.

1. **In the cell below**, type your group's track number after `TRACK =` (for
   example `TRACK = 2`) and run it. You should see the file it loaded and its main
   column.
2. `VERIFY` **Run the cell after it.** It prints the two functions you just loaded. **Read
   every line:** you are responsible for what they do.
3. `VERIFY` **Run the chain cell.** You should see real row counts at ingest and clean, the
   line *still the fake skeleton: analyse, decide, respond* (those come next), and
   still 0 events.

In [ ]:
TRACK = None     # TODO(you): type your group's track number, 1 to 6, for example TRACK = 2
KIT = {1: ("smart_money", "smart_money_ohlcv.csv", "Close"),
       2: ("smart_building", "smart_building_sensors.csv", "co2_ppm"),
       3: ("smart_farm", "sg_rainfall_daily.csv", "rain_mm_mean"),
       4: ("smart_city", "sg_taxi_hourly.csv", "taxi_count"),
       5: ("smart_self", "smart_self_daily.csv", "steps"),
       6: ("free_form", "example_data.csv", "value")}
if TRACK not in KIT:
    print("Type your track number after TRACK = (1 to 6), then run this cell again.")
else:
    kit_name, data_file, VALUE_COL = KIT[TRACK]
    fetch("track_kit.py", data_file)
    import track_kit
    ingest = getattr(track_kit, "ingest_" + kit_name)
    clean = getattr(track_kit, "clean_" + kit_name)
    print(f"Track {TRACK}: loaded ingest and clean for {data_file}; main column VALUE_COL = '{VALUE_COL}'")

In [ ]:
import inspect
if TRACK in KIT:
    print(inspect.getsource(ingest)); print(inspect.getsource(clean))

In [ ]:
sent = pipeline()      # the chain, with your real ingest and clean

**Stage 3: analyse.** It takes the cleaned table and adds a True/False column
`flag` marking unusual readings: the rolling z-score from Lab 5.

**Step 1.** `PROMPT` **Ask Gemini.** Open Gemini (the **spark icon** at the bottom centre of the Colab window), start a new chat (click **⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**), paste this and press Enter. To copy it, drag
across the quoted text below from its first word to its last, then press Ctrl+C
(Cmd+C on a Mac):

> **Rules:** Return every row it receives: do not drop rows. Only define the function;
> do not call it. Explain in plain words what it does.
>
> **Task:** Write a Python function `analyse(df)`. `df` has a `timestamp` column and a
> numeric column whose name is stored in the variable `VALUE_COL`. Return a copy of df
> with a new True/False column `flag`, True where the value is more than 3 standard
> deviations from the mean of the previous 24 rows.

**Step 2.** `GENERATE` **Accept its code.** Gemini explains its plan in the chat and puts its code
into the notebook as a **green cell**. Click **Accept and run**: the button sits at
the bottom of the chat panel, just above the box where you type. It does not matter
where in the notebook the green cell lands. Ignore the blue suggestion chips under
its answer. If the code appears only in the chat, click the copy icon on the code,
click once on this text, click **+ Code** at the top of the page, paste into the new
cell, and run it. If Gemini ever shows red and green lines inside the check cell
(first line `# TODO(you)`), click **Cancel**: check cells are never changed.
Colab scrolls to the green cell, even to the very top of the notebook, so after
**Accept and run** scroll back down to this task for Step 3.

**Step 3.** `VERIFY` **Run the chain cell below.** You should see the same row count at analyse as at clean, and no `analyse` in the *still the fake skeleton* line. The decide line may now show hundreds of events and *would send* hundreds of alerts: that is still the fake decide, which Stage 4 replaces.

**Step 4.** `REFINE` **Only if a check fails, a number looks wrong, or Gemini's green cell
shows red error text:** in the same chat, type one
sentence saying what you saw and what you expected, for example *analyse returned fewer rows than it received; return every row and only add the flag column.* Press Enter,
click **Accept and run** (Gemini may add a new green cell or change its own earlier
one; either is fine), and run the chain cell below again. If everything
passed, there is nothing to fix: move on.

In [ ]:
# TODO(you): Accept and run Gemini's green cell first (wherever it landed) (it defines analyse), then run this cell.
sent = pipeline()

**Stage 4: decide.** It keeps only the flagged rows worth a message: debounced
and rate-limited, as in Lab 7.

**Step 1.** `PROMPT` **Ask Gemini.** Open Gemini (the **spark icon** at the bottom centre of the Colab window), start a new chat (click **⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**), paste this and press Enter. To copy it, drag
across the quoted text below from its first word to its last, then press Ctrl+C
(Cmd+C on a Mac):

> **Rules:** Only define the function; do not call it. Explain in plain words what it
> does.
>
> **Task:** Write a Python function `decide(df)`. `df` has a `timestamp` column and a
> True/False column `flag`. Return only the rows where flag is True and the row before
> is also True (a held condition), keeping at most one row per calendar day.

**Step 2.** `GENERATE` **Accept its code.** Gemini explains its plan in the chat and puts its code
into the notebook as a **green cell**. Click **Accept and run**: the button sits at
the bottom of the chat panel, just above the box where you type. It does not matter
where in the notebook the green cell lands. Ignore the blue suggestion chips under
its answer. If the code appears only in the chat, click the copy icon on the code,
click once on this text, click **+ Code** at the top of the page, paste into the new
cell, and run it. If Gemini ever shows red and green lines inside the check cell
(first line `# TODO(you)`), click **Cancel**: check cells are never changed.
Colab scrolls to the green cell, even to the very top of the notebook, so after
**Accept and run** scroll back down to this task for Step 3.

**Step 3.** `VERIFY` **Run the chain cell below.** You should see at least 1 event at decide, and no `decide` in the *still the fake skeleton* line. Any number from 1 up is fine: there is no single right count, because it depends on your track's data. Zero is the only result to fix (Step 4). If it is still zero after Step 4, put your hand up and show the tutor this cell: on some tracks the analyse rule from Stage 3 needs changing first.

**Step 4.** `REFINE` **Only if a check fails, a number looks wrong, or Gemini's green cell
shows red error text:** in the same chat, type one
sentence saying what you saw and what you expected, for example *decide returned 0 events; keep a row when it is True and the row before it is also True, using the flag column from analyse.* Press Enter,
click **Accept and run** (Gemini may add a new green cell or change its own earlier
one; either is fine), and run the chain cell below again. If everything
passed, there is nothing to fix: move on.

In [ ]:
# TODO(you): Accept and run Gemini's green cell first (wherever it landed) (it defines decide), then run this cell.
sent = pipeline()

**Stage 5: respond.** It turns each event into a message a person can act on.

**Step 1.** `PROMPT` **Ask Gemini.** Open Gemini (the **spark icon** at the bottom centre of the Colab window), start a new chat (click **⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**), paste this and press Enter. To copy it, drag
across the quoted text below from its first word to its last, then press Ctrl+C
(Cmd+C on a Mac):

> **Rules:** Only define the function; do not call it. Explain in plain words what it
> does.
>
> **Task:** Write a Python function `respond(events)`. `events` is a DataFrame with a
> `timestamp` column and a column whose name is stored in `VALUE_COL`. Return a Python
> list with one message per row, such as "co2_ppm was 1240 at 14:35 on 3 Jun: unusual
> for this time, please check", using each row's own value and time. Print how many
> messages there are.

**Step 2.** `GENERATE` **Accept its code.** Gemini explains its plan in the chat and puts its code
into the notebook as a **green cell**. Click **Accept and run**: the button sits at
the bottom of the chat panel, just above the box where you type. It does not matter
where in the notebook the green cell lands. Ignore the blue suggestion chips under
its answer. If the code appears only in the chat, click the copy icon on the code,
click once on this text, click **+ Code** at the top of the page, paste into the new
cell, and run it. If Gemini ever shows red and green lines inside the check cell
(first line `# TODO(you)`), click **Cancel**: check cells are never changed.
Colab scrolls to the green cell, even to the very top of the notebook, so after
**Accept and run** scroll back down to this task for Step 3.

**Step 3.** `VERIFY` **Run the chain cell below.** You should see the number of messages printed at the end of the chain, and no stage left in the *still the fake skeleton* line.

**Step 4.** `REFINE` **Only if a check fails, a number looks wrong, or Gemini's green cell
shows red error text:** in the same chat, type one
sentence saying what you saw and what you expected, for example *respond returned an empty list although decide found events; make one message per row of events.* Press Enter,
click **Accept and run** (Gemini may add a new green cell or change its own earlier
one; either is fine), and run the chain cell below again. If everything
passed, there is nothing to fix: move on.

In [ ]:
# TODO(you): Accept and run Gemini's green cell first (wherever it landed) (it defines respond), then run this cell.
sent = pipeline()

**Optional: send the first message to your phone.** You set up Telegram in Lab 7
and the token is still in Colab's Secrets (key icon). Run the next cell; you should
see *sent* and the message arrive within seconds. If a box asks to give this notebook
access to the secret, click **Grant access**. If the cell prints *Telegram not set up
here*, click the key icon and switch on **Notebook access** for TELEGRAM_TOKEN, then
run it again. This step is optional: if it still does not send, move on.

In [ ]:
import requests
try:
    from google.colab import userdata
    TOKEN = userdata.get("TELEGRAM_TOKEN")
    chats = [u["message"]["chat"]["id"] for u in requests.get(f"https://api.telegram.org/bot{TOKEN}/getUpdates", timeout=20).json().get("result", []) if "message" in u]
    if chats and sent:
        reply = requests.get(f"https://api.telegram.org/bot{TOKEN}/sendMessage", params={"chat_id": chats[-1], "text": sent[0]}, timeout=20).json()
        print("sent:", sent[0] if reply.get("ok") else reply.get("description"))
    else:
        print("Nothing to send yet, or no chat found: send your bot a message in Telegram and run this cell again.")
except Exception as e:
    print("Telegram not set up here:", e, "- this step is optional.")

## Task 3 — Verify the whole chain · `VERIFY`

The contract below checks the things that break at the joins.

0. `VERIFY` **Run the cell just below first.** It prints the numbers your explanation needs:
   rows out of ingest and out of clean, rows removed (with the reason), and rows
   added (with the reason).
1. **In the check cell after it**, replace the words between the quotes after
   `EXPLANATION =` with your explanation of any change in the row count between
   ingest and clean, as one line of text between the quotes. Use only the lines
   starting *rows removed*, *rows added* and *other rows removed*, and skip any whose
   number is 0. For each, write the number, the word removed or added, and the words
   in its brackets as the reason, joined with a semicolon, exactly like this (Track 2's
   numbers): *"140 rows removed: the same time written twice; 66 rows added: clean put
   the data on a regular timetable, and gaps became blank rows"*. If a line says *read
   clean's code*, ask Gemini in a new chat *Which rows does clean() remove, and
   why? One sentence.* and use its answer as the reason. Use your own track's numbers. This line is yours to type; only the lines under
   `# ---- verification contract` are locked.
   If the two counts are equal, write why nothing changed, for example *"no rows
   changed: this file has no duplicates or sentinels"*.
2. `VERIFY` **Run the cell.** You should see two PASS lines and a bar chart. If
   `response_fired` fails and the Stage 4 chain cell says 0 events at decide, start a
   new chat (click **⋮** at the top right of the Gemini panel, then **New chat**,
   then **Yes**) and type *My decide function in this notebook returned 0 events;
   keep a row when it is True and the row before it is also True, using the flag
   column from analyse.* Press Enter, click **Accept and run**, run the Stage 4 and
   Stage 5 chain cells again, then this cell.

In [ ]:
_r = ingest(); _c = clean(_r)
_dups = int(_r["timestamp"].duplicated().sum())
_added = len(_c) - (len(_r) - _dups)
print(f"rows out of ingest: {len(_r):,}   rows out of clean: {len(_c):,}   change: {len(_c) - len(_r):+,}")
print(f"rows removed: {_dups} duplicate timestamps (the same time written twice)")
print(f"rows added  : {max(_added, 0)} empty time slots (clean put the data on a regular timetable, and gaps became blank rows)")
if _added < 0:
    print(f"other rows removed: {-_added} (read clean's code above to see which, and why)")

In [ ]:
raw_df   = ingest()
clean_df = clean(raw_df)

# TODO(you): explain any change in row count between the two stages
EXPLANATION = "TODO(you): explain any change in row count through your pipeline"

# ---- verification contract — do not edit ----
check.rows_accounted_for(
    raw_df, clean_df,
    names=["ingest", "clean"],
    explain=EXPLANATION,
)
check.response_fired(sent, minimum=1)
check.report()

show.stages({"ingest": len(raw_df), "clean": len(clean_df)}, title="Rows at each stage boundary")

**What you should see:** two bars, and a difference between them that your
`EXPLANATION` accounts for line by line. If the bars are equal and you cleaned
anything, the cleaning did not run on the data that reached this stage.

## Task 4 — Draw it

1. **Draw** five boxes on the Lab 8 sheet (or on paper; photograph it for CA3), one
   per stage, joined by four arrows.
2. **Run the cell below.** For each stage it prints the rows, the columns and the
   time between rows.
3. **Write on each arrow** what passes between the stages: **how many rows, how
   often (the cadence), which columns**, copied from that printout.

In [ ]:
_t = ingest(); _c = clean(_t); _a = analyse(_c); _d = decide(_a)
for name, d in [("ingest -> clean", _t), ("clean -> analyse", _c), ("analyse -> decide", _a), ("decide -> respond", _d)]:
    gap = d["timestamp"].diff().median() if "timestamp" in d and len(d) > 1 else "n/a"
    print(f"{name:18s} rows {len(d):6,}   every {gap}   columns {list(d.columns)}")

This diagram goes into your CA3 report, so it is not throwaway work.

---
## Predict: Which day will be wettest? (15 minutes)

*The Prediction League: one question per lab, settled by reality. Individual work: you enter on your own, under your own name.*

**The question:** Which day of the target week has the most island-average rain: 1 for Monday through 7 for Sunday. The target is the week of Monday 2027-01-18.

**How it works:** you enter a short function, `predict(history)`, not a number. When
entries close, the league runs your function once on the latest data. When the target
is over, the real value is compared with yours. League page (with the closing time):
https://stanleysp-eng.github.io/EP0418-labs/predict.html?lab=8

**Step 1.** **Run the cell below.** It prints the number from the simple rule,
*the weekday with the most rain on average over the last nine weeks*. That is the number to beat.

In [ ]:
fetch("sg_rainfall_daily.csv")
rain = pd.read_csv("sg_rainfall_daily.csv", parse_dates=["date"])

def predict(history):
    # the baseline: the weekday with the most rain on average over the last nine weeks
    recent = history.tail(63)
    by_day = recent.groupby(recent["date"].dt.dayofweek)["rain_mm_mean"].mean()
    print("average mm, Monday to Sunday:", by_day.round(1).tolist())
    return float(by_day.idxmax() + 1)

print("The baseline predicts", round(predict(rain), 1), "(1 = Monday ... 7 = Sunday)")

**What you should see:** *The baseline predicts 3.0 (1 = Monday ... 7 = Sunday)*.

**Step 2.** **Ask Gemini to write your predict.** Start a new
chat (**⋮** › **New chat** › **Yes**), paste the prompt and press Enter.

> **Rules:** Put it in a new cell and do not change the cells above. Write only the function and any import lines it needs. Keep the name *predict* and its one input, *history*, a pandas table with the columns date, rain_mm_mean (island average, mm), rain_mm_max (wettest gauge, mm) and stations_reporting, oldest first. Return one number with float(...). **Task:** Write a predict(history) that estimates this: which day of the target week has the most island-average rain: 1 for Monday through 7 for Sunday. Do it in a way you think beats the simple rule (the weekday with the most rain on average over the last nine weeks). Explain the idea in one sentence.

Click **Accept and run** on Gemini's green cell. It prints nothing; it only defines the
function. *Red and green lines inside the Step 1 cell?* Click **Cancel** and type *put it in a new cell*.

**Step 3.** **Run the cell below.**

In [ ]:
# TODO(you): run Gemini's green cell first (Step 2)
show_predict(rain, "(1 = Monday ... 7 = Sunday)")

**What you should see:** *Your predict says* and a number, usually between 1 and 7, then your
code between two dashed lines. Output says *Gemini's predict has not run yet*? Do what that message says.
*Red error?* Paste the last red line into Gemini with *fix this*, click
**Accept and run**, and run this cell again.

<details><summary><b>Using ChatGPT, Claude or another AI instead of Colab?</b> Open this for a prompt to copy</summary>

That AI cannot see this notebook, so Step 2's prompt is not enough for it. This one spells
out every rule the league checks. Replace the *My idea* line with your idea,
paste it into the AI, then do Step 4 with the AI's code instead of the Step 3 output.
Want the AI to look at the numbers first? Download the data,
https://stanleysp-eng.github.io/EP0418-labs/data/rain_daily.csv (the same file your
function receives, updated every night), and attach it to the chat with the prompt.

```text
I am entering a prediction contest. Write one Python function for it.

The question: Which day of the target week has the most island-average rain: 1 for Monday through 7 for Sunday. The target is the week of Monday 2027-01-18 to Sunday 2027-01-24.
The simple rule to beat: the weekday with the most rain on average over the last nine weeks.
My idea: [write your idea here in a few words, or delete this line]

The contest runs the function automatically, so follow these rules exactly:
1. Name it predict, with exactly one input called history:  def predict(history):
2. history is a pandas DataFrame that the contest passes in, oldest row first. Its column names are exactly: date, rain_mm_mean, rain_mm_max, stations_reporting. What they hold: date, rain_mm_mean (island average, mm), rain_mm_max (wettest gauge, mm) and stations_reporting. The date column is a pandas datetime, one row per day, with no time zone.
3. Do not read any file and do not ask for input. All the data is already in history. If I attach a data file, look at it only to choose the idea; the function must still take everything from history.
4. Return one number with return float(...). Not a list, a table, text, None or NaN.
5. Use only pandas, numpy and the Python standard library, and finish in under 30 seconds.
6. Give only the import lines and the function: no example data, no test code, nothing that calls predict.

After the code, explain the idea in one sentence.
```
</details>

**Step 4. Submit on the league page.**
1. **Copy your code** from the Step 3 output: everything between the two dashed lines.
2. **Open** https://stanleysp-eng.github.io/EP0418-labs/predict.html?lab=8 in a new tab and type your name as on the class list (the same name every week).
3. **Paste** it into the **Code** box: click in it, Ctrl+A, Ctrl+V (Cmd on a Mac).
4. **Click Run tests** (about 15 seconds the first time). **What you should see:** *All 4 cases ran* and four tabs: three past cases and *Your entry*. On *Your entry*, Actual says *not known yet*, and the number differs from Colab's because the page uses newer data: both are normal. A red *missed by* only means the simple rule did better on that case. A **red heading** instead of *All 4 cases ran* is an error: copy the red text under the case marked red (or under the heading), paste it into the same Gemini chat with *fix this*, click **Accept and run**, run Step 3 again, then paste and **Run tests** again.
5. **Click Submit.** *Submitted at ...* confirms it. Submit again any time before the close; the latest one counts.

**Write it down** (on the sheet, or double-click here): your function's number (Step 3),
the baseline (Step 1), and the one-sentence idea Gemini gave. After the target, the
real answer is on the league page.

**My number / the baseline / the idea:**

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

There is no single right answer here. A good entry gives all three parts: your
function's number, the baseline (3.0), and an idea that uses the data. By weekday, Wednesday averaged the most rain in the file (7.9 mm) and Sunday the least (2.3 mm), but sixty days hold only about eight of each weekday, so one storm decides the order. This question is close to luck; a good entry says so and still uses the data, for example the wettest weekday over the most recent weeks.
The common weak answer is "Gemini wrote it". That says who, not what. The past-case
tabs on the league page show whether the idea would have beaten the simple rule
lately; three cases are too few to prove it.
</details>

---
## Read before you run

A student replaced the analyse stage of their skeleton with the code below. The pipeline still ran end to end without an error.

The code is switched off: every line starts with `#`, so it cannot run yet.

1. **Read it. Do not run it.** Answer questions 1 and 2 in the box under the code
   cell, or on the sheet.
2. **Ask Gemini to explain it.** Click in the code cell, press Ctrl+A then Ctrl+C
   (Cmd on a Mac). In Gemini, start a new chat (**⋮** › **New chat** › **Yes**),
   paste the code into the chat box, add *Explain this code in plain words.* and
   press Enter. If Gemini offers a green cell or red and green lines, click
   **Cancel**: this step only explains. Did the explanation point out the problem?
   Often it does not.
3. **Then run it.** Delete the `# ` at the start of each line below the first line,
   run the cell, and see whether you were right.

**Question 1.** The Task 1 skeleton's `ingest()` returned ten rows with `value` 0 to 9. How many rows come out of this stage? Your decide stage looks at “the row before”. After this change, what is the row before the row with value 6? Answer with a value, or none if there is no row before it.

**Question 2.** Write down the rows in and rows out, and why the decide stage's “row before” is no longer the reading just before in time.

<details><summary>Hint 1: where to look</summary>

What happens to the rows whose value is 5 or less? Then ask what "the previous row" means in the next stage.

</details>
<details><summary>Hint 2: closer to the answer</summary>

Ten rows in, four out (values 6 to 9). Values 0 to 5 are gone, so the row before 6 is now none, although in time the reading before it was 5. "Two consecutive readings" in stage 4 now means two consecutive *flagged* readings, which is a different thing.

</details>

An explanation that reads well is not evidence the code is right. 

**Optional, to fix it:** in the same chat, type *Fix the problem in this code that
Hint 2 describes. Say in one sentence what you changed.* If that one change matches
Hint 2, click **Accept and run** (accepting is right here: you asked for the fix). If
it changed other things too, click **Cancel** and type *fix only that, nothing else*.
You should see: the count is now 10, not 4: every row comes back, with `flag` True or False.

In [ ]:
# ---- Every line below this one is the code. Do steps 1 and 2 first. For step 3, delete the "# " at the start of each line below, then run the cell. ----
# def their_analyse(df):          # their version, under its own name so yours is untouched
#     return df[df.value > 5].assign(flag=True)
# print("rows out of the Task 1 skeleton's ten:", len(their_analyse(pd.DataFrame({"value": range(10)}))))

**Answer to question 1:**

**Answer to question 2:**

**Did Gemini's explanation mention the fault? (yes / no, and the sentence):**

**Was I right, once I ran it?**

---
## Quiz yourself before you leave (5 minutes)

The checks test the code. This tests **you**. In Gemini, start a new chat
(**⋮** › **New chat** › **Yes**), paste this into the chat box and press Enter:

> **Rules:** Ask one short question at a time, in plain words. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer.
>
> **Task:** Quiz me on this lab: five questions on these ideas: the five stages and what passes between them; why two correct stages can fail at the join; what the row count tells you at each stage boundary; what rows_accounted_for and response_fired check end to end; where in the pipeline your project's biggest risk sits. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks one question at a time in the chat panel; answer in the chat box, in
your own words, without looking back. Ignore the suggestion chips. Using ChatGPT,
Claude or Copilot instead? Delete the sentence *Take the examples from the code and
outputs in this notebook.* first: they cannot see your notebook.

Then write your score and the idea to revise below. Your tutor asks for them.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

*(Your answer box: double-click this cell. Type your answer here.)*

---
## Before you leave

Double-click the box cell below and type there, or write on the last page of the Lab 8
sheet:

- **Mistake caught:** which join broke first when you swapped a stage in, and the
  mismatched assumption. If nothing broke, one thing Gemini got wrong today. If Gemini got nothing wrong
  either, write the stage where you expected a join to break, and the row count
  that showed it did not.
- **Decision:** what passes between your stages (cadence, columns) and why. To draft
  it, ask Gemini in a new chat *Summarise what my analyse, decide and respond
  functions do, in three sentences, and give the row or event count each one
  printed in the last chain cell.* Compare each number in its answer with the
  last chain cell's printout (the lines *analyse ->*, *decide ->* and the message
  count); where one differs, change it to the printed number. Paste the result below.
  Using another AI? It cannot see your notebook: paste your analyse, decide and
  respond cells and the last chain cell's printout under the request.

**Mistake caught:** *(what I expected · what was there · how I found it. If Gemini made no mistake today, use the other option in the text just above this box.)*

**Decision:** *(what I decided · why · the evidence)*

*(Your answer box: double-click this cell. Type your answer here.)*